In [1]:
import os
from dlisio import dlis
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
from db import db

# Loading wells

Each well is a folder, named by its directory name. Inside each well folder, there are multiple DLIS files. Each DLIS file is loaded and processed to extract its frames and channels, which are then stored in the database.

In [3]:
wells = {}
with os.scandir("wells") as entries:
    for entry in entries:
        path = entry.path
        wells[entry.name] = [f for f in os.listdir(path)]

In [4]:
LOG_TYPES = {
  "MUDLOG": 1,
  "MWD": 2
}
db.find_or_create("log_type", {"name": "MUDLOG"})
db.find_or_create("log_type", {"name": "MWD"})

2

In [ ]:
def parse_well(well_name):
    well_id = db.find_or_create("well", {"name": well_name})

    for file in wells[well_name]:
        if not file.upper().endswith(".DLIS"):
            continue

        with dlis.load(f"wells/{well_name}/{file}") as (f, *tail):
            log_type_id = LOG_TYPES["MWD"]  # Default log type

            if "MUD" in file.upper():
                log_type_id = LOG_TYPES["MUDLOG"]

            for frame in f.frames:
                file_id = db.find_or_create(
                    "logical_file",
                    {
                        "well_id": well_id,
                        "name": file,
                        "log_type_id": log_type_id,
                        "index_type": frame.index_type,
                    },
                )

                curves = frame.curves()

                if curves.size == 0 or frame.index not in curves.dtype.names:
                    continue

                index_array = curves[frame.index]

                frame_readings = []

                for c in frame.channels:
                    if c.name == frame.index:
                        continue

                    if c.name not in curves.dtype.names:
                        print(f"⚠️ Channel '{c.name}' is declared in frame metadata but has no data rows. Skipping.")
                        continue

                    channel_id = db.find_or_create("channel", {"name": c.name, "unit": c.units})
                    channel_data = curves[c.name]

                    for index_val, reading_val in zip(index_array, channel_data):
                        frame_readings.append((
                            file_id, 
                            channel_id, 
                            float(index_val), 
                            str(reading_val) 
                        ))

                if frame_readings:
                    df = pd.DataFrame(
                        frame_readings,
                        columns=[
                            "logical_file_id",
                            "channel_id",
                            "index_value",
                            "reading_value",
                        ],
                    )

                    db.bulk_insert_readings(df)

In [6]:
for well_name in wells.keys():
    parse_well(well_name)

ValueError: no field of name SWF2

In [7]:
db.close()